In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data = pd.read_csv('/content/drive/MyDrive/YOUTUBE/Final_Mean.csv')

In [ ]:
data

,Unnamed: 0,FaceRectX,FaceRectY,FaceRectWidth,FaceRectHeight,FaceScore,x_0,x_1,x_2,x_3,...,disgust,fear,happiness,sadness,surprise,neutral,input,frame,question,answer
0,0,291.363858,35.447479,93.662453,119.462808,0.999587,290.202567,289.394182,289.544206,291.524437,...,0.002833,0.001497,0.857664,0.006462,0.019306,0.109500,/content/data/frame0.jpg,0,Are there signs of Unusual/ excessive smiling?,no
1,1,291.363858,35.447479,93.662453,119.462808,0.999587,290.202567,289.394182,289.544206,291.524437,...,0.002833,0.001497,0.857664,0.006462,0.019306,0.109500,/content/data/frame0.jpg,0,Does the child have reduced eye contact?,no
2,2,291.363858,35.447479,93.662453,119.462808,0.999587,290.202567,289.394182,289.544206,291.524437,...,0.002833,0.001497,0.857664,0.006462,0.019306,0.109500,/content/data/frame0.jpg,0,Are their lips parted?,no
3,3,291.363858,35.447479,93.662453,119.462808,0.999587,290.202567,289.394182,289.544206,291.524437,...,0.002833,0.001497,0.857664,0.006462,0.019306,0.109500,/content/data/frame0.jpg,0,Is the face asymmetric(FA)?,no
4,4,291.363858,35.447479,93.662453,119.462808,0.999587,290.202567,289.394182,289.544206,291.524437,...,0.002833,0.001497,0.857664,0.006462,0.019306,0.109500,/content/data/frame0.jpg,0,Are cheeks puffed or raised with happiness?,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343137,343137,905.481011,339.867313,303.741754,395.575138,0.998284,897.603317,895.630486,897.173160,901.920121,...,0.000766,0.001781,0.087313,0.029797,0.004213,0.844134,/content/data/frame604.jpg,0,Is the face asymmetric(FA)?,no
343138,343138,905.481011,339.867313,303.741754,395.575138,0.998284,897.603317,895.630486,897.173160,901.920121,...,0.000766,0.001781,0.087313,0.029797,0.004213,0.844134,/content/data/frame604.jpg,0,Are cheeks puffed or raised with happiness?,no
343139,343139,905.481011,339.867313,303.741754,395.575138,0.998284,897.603317,895.630486,897.173160,901.920121,...,0.000766,0.001781,0.087313,0.029797,0.004213,0.844134,/content/data/frame604.jpg,0,"Are there any noticeable lip movements, (such ...",no
343140,343140,905.481011,339.867313,303.741754,395.575138,0.998284,897.603317,895.630486,897.173160,901.920121,...,0.000766,0.001781,0.087313,0.029797,0.004213,0.844134,/content/data/frame604.jpg,0,Does the lip corner depressor show discomfort ...,no


## Transformer

In [ ]:
! pip install transformers

In [ ]:
import transformers

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import tokenizer_from_json

In [ ]:
data = data.dropna()

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.optimizers import Adam
from transformers import TFAutoModel, AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tensorflow.keras.callbacks import EarlyStopping

# Assuming 'data' is a DataFrame containing 'question', 'answer', and features

# Split the data into training and validation sets
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

# Tokenize the text using transformer's tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
question_sequences_train = tokenizer(list(train_data['question']), padding='longest', truncation=True, return_tensors='tf')
question_sequences_val = tokenizer(list(val_data['question']), padding='longest', truncation=True, return_tensors='tf')

# Prepare input features
features_train = train_data[[
    'AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU07', 'AU09', 'AU10', 'AU11', 'AU12',
    'AU14', 'AU15', 'AU17', 'AU20', 'AU23', 'AU24', 'AU25', 'AU26', 'AU28', 'AU43',
    'anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise', 'neutral'
]].values

features_val = val_data[[
    'AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU07', 'AU09', 'AU10', 'AU11', 'AU12',
    'AU14', 'AU15', 'AU17', 'AU20', 'AU23', 'AU24', 'AU25', 'AU26', 'AU28', 'AU43',
    'anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise', 'neutral'
]].values

# Prepare target labels
labels_train = np.array([1 if ans.lower() == 'yes' else 0 for ans in train_data['answer']])
labels_val = np.array([1 if ans.lower() == 'yes' else 0 for ans in val_data['answer']])

# Load the transformer model
transformer_model = TFAutoModel.from_pretrained('bert-base-uncased')

# Define the custom metric function for AUC-ROC
def auc_roc(y_true, y_pred):
    return tf.py_function(roc_auc_score, (y_true, y_pred), tf.float64)

# Define the model architecture
input_ids = Input(shape=(None,), dtype=tf.int32)
attention_mask = Input(shape=(None,), dtype=tf.int32)
input_features = Input(shape=(27,))

embeddings = transformer_model([input_ids, attention_mask])[0]
pooled_output = tf.reduce_mean(embeddings, axis=1)

concat = Concatenate()([pooled_output, input_features])
output = Dense(1, activation='sigmoid')(concat)

model = Model(inputs=[input_ids, attention_mask, input_features], outputs=output)

# Compile the model with AUC-ROC metric
model.compile(optimizer=Adam(learning_rate=2e-5), loss='binary_crossentropy', metrics=['accuracy', auc_roc])

# Use EarlyStopping callback to stop training if AUC-ROC doesn't improve
early_stopping = EarlyStopping(monitor='val_auc_roc', patience=3, restore_best_weights=True)

# Train the model
model.fit(
    [question_sequences_train['input_ids'], question_sequences_train['attention_mask'], features_train],
    labels_train,
    epochs=3,
    batch_size=32,
    validation_data=([question_sequences_val['input_ids'], question_sequences_val['attention_mask'], features_val], labels_val),
    callbacks=[early_stopping]
)

# Save the model
model.save('transformer_model_with_auc_roc.h5')


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Epoch 1/3


  73/8404 [..............................] - ETA: 30:22:43 - loss: 0.6380 - accuracy: 0.6327 - auc_roc: 0.6184